# 13 — Stacker Experiment: Does LightGBM's Diversity Pay Off?

Notebook 12 produced a striking result: LightGBM scored 0.865 against the best NN anchor's 0.957 (a 9.2-point AUC gap), but its per-sample top-class probabilities correlated only **0.04 to 0.24** with the NN anchors' — far below the typical 0.90+ seen inside the NN family. That gap was wide enough for notebook 12's verdict logic to DROP LightGBM as a base-model candidate. The correlation was extreme enough that the verdict probably shouldn't have applied to the *ensemble* question. This notebook tests it.

**Hypothesis.** Adding LightGBM as a 5th base model to the stacker shootout from notebook 11 produces a measurable val-AUC lift on at least one blend method — most likely rank-mean (which neutralizes the calibration difference between NN softmaxes and LGBM probabilities) or per-class logistic regression.

**Decision criterion.** Lift ≥ 0.001 averaged over 3 random seeds counts as a real signal at these correlation levels. Anything under is split-noise. Lift ≥ 0.003 is the threshold for actually adding LightGBM to your submission ensemble; the engineering cost (one extra feature pipeline) is real and the gain has to clear it.

**Structure.** Same stacker methods as notebook 11, run twice: once with NN-only (baseline), once with NN + LGBM. Repeated across 3 seeds. Report the lift distribution per method.


## 0 — Setup

In [ ]:
import os, sys, json, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

META_CSV       = ROOT / 'data' / 'raw' / 'train.csv'
FOLDS_CSV      = ROOT / 'data' / 'folds' / 'folds.csv'
EXP_DIR        = ROOT / 'experiments'
FEATURES_CACHE = ROOT / 'experiments' / '_classical_features.feather'
LGBM_OOF_PATH  = ROOT / 'experiments' / 'classical_lightgbm' / 'oof_preds.csv'
FOLD_FOR_VAL   = 0
N_SEEDS        = 3       # seeds for stacker fit splits — averages out split noise
EPS = 1e-7


def macro_auc_skip_empty(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        pos = y_true[:, c].sum()
        if 0 < pos < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return (float(np.mean(aucs)) if aucs else float('nan')), len(aucs)


def rank_normalize(p):
    out = np.zeros_like(p, dtype=np.float64)
    for c in range(p.shape[1]):
        out[:, c] = rankdata(p[:, c]) / len(p)
    return out

print('setup OK')


## 1 — Load NN anchor OOFs

Same loader pattern as notebook 11 — concatenates per-fold OOFs, then uses the column reconciliation cell from the patched version. We don't worry about effb0's 468-column issue here because notebook 12's diversity check showed effb0's correlation column was NaN — meaning the diversity argument is fully supported by the other three NN anchors. We drop effb0 from this experiment by default.


In [ ]:
NN_MODELS = {
    # 'effb0':       'baseline_effb0/baseline_effb0',           # disabled: column-name bug
    'effv2s':      'effv2s_finetune/effv2s_finetune',
    'convnext':    'sed_finetune/sed_finetune',
    'seresnext':   'seresnext_finetune/seresnext_finetune',
    'effv2s_focal': 'effvs2_focal/effv2s_focal',
}
N_FOLDS = 5


def load_oof(pattern, n_folds=N_FOLDS):
    parts = []
    for f in range(n_folds):
        for sub in [f'{pattern}_fold{f}/oof_preds.csv', f'{pattern}/fold{f}/oof_preds.csv']:
            p = EXP_DIR / sub
            if p.exists():
                parts.append(pd.read_csv(p))
                break
        else:
            return None
    return pd.concat(parts, ignore_index=True)


loaded_nn = {n: load_oof(p) for n, p in NN_MODELS.items()}
loaded_nn = {n: d for n, d in loaded_nn.items() if d is not None}
for n, d in loaded_nn.items():
    print(f'  {n:<14} {len(d):>6} rows, {d.shape[1]-1} cols')


## 2 — Generate the LightGBM OOF

Loads from `experiments/classical_lightgbm/oof_preds.csv` if it exists; otherwise re-trains from the feature cache produced by notebook 12 (`experiments/_classical_features.feather`). Re-training is ~80 minutes — only happens once, then the OOF is saved for reuse.

The OOF is saved with **species code** column headers matching the canonical NN species set, so the reconciliation in section 3 handles alignment cleanly. Species not present in the training data (in our case, 28 of the 234) get a constant 0.5 (neutral prior) — they'll be skipped by the macro-AUC metric anyway if they have no val positives.


In [ ]:
def train_lightgbm_oof(feat_df, folds_df, fold_for_val=FOLD_FOR_VAL):
    """Per-class OvR LightGBM, trained on folds != fold_for_val, predicting fold_for_val."""
    if not HAS_LGB:
        raise RuntimeError('lightgbm not installed')
    df = feat_df.merge(folds_df[['filename', 'fold']], on='filename', how='left')
    train_mask = (df['fold'] != fold_for_val).values
    val_mask   = (df['fold'] == fold_for_val).values
    feat_cols = [c for c in df.columns if c.startswith('f') and c[1:].isdigit()]
    X_tr = df.loc[train_mask, feat_cols].values.astype(np.float32)
    X_va = df.loc[val_mask,   feat_cols].values.astype(np.float32)
    species_list = sorted(df['primary_label'].dropna().unique())
    sp_to_idx = {sp: i for i, sp in enumerate(species_list)}
    Y_tr = np.zeros((train_mask.sum(), len(species_list)), dtype=np.float32)
    for i, sp in enumerate(df.loc[train_mask, 'primary_label'].values):
        if isinstance(sp, str) and sp in sp_to_idx:
            Y_tr[i, sp_to_idx[sp]] = 1.0
    preds = np.full((val_mask.sum(), len(species_list)), 0.5, dtype=np.float32)
    t0 = time.time()
    for c in range(len(species_list)):
        yc = Y_tr[:, c]
        if yc.sum() < 5:
            continue
        clf = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05,
                                 min_data_in_leaf=20, class_weight='balanced',
                                 n_jobs=1, verbosity=-1, force_col_wise=True)
        clf.fit(X_tr, yc)
        preds[:, c] = clf.predict_proba(X_va)[:, 1]
        if (c + 1) % 25 == 0:
            print(f'    {c+1}/{len(species_list)} species  elapsed={time.time()-t0:.0f}s')
    out = pd.DataFrame(preds, columns=species_list)
    out.insert(0, 'filename', df.loc[val_mask, 'filename'].values)
    return out


if LGBM_OOF_PATH.exists():
    print(f'loading cached LightGBM OOF from {LGBM_OOF_PATH}')
    lgbm_oof = pd.read_csv(LGBM_OOF_PATH)
elif FEATURES_CACHE.exists() and FOLDS_CSV.exists() and HAS_LGB:
    print(f'no cached OOF — retraining LightGBM from {FEATURES_CACHE}. This takes ~80 min.')
    feat_df  = pd.read_feather(FEATURES_CACHE)
    folds_df = pd.read_csv(FOLDS_CSV)
    lgbm_oof = train_lightgbm_oof(feat_df, folds_df)
    LGBM_OOF_PATH.parent.mkdir(parents=True, exist_ok=True)
    lgbm_oof.to_csv(LGBM_OOF_PATH, index=False)
    print(f'saved → {LGBM_OOF_PATH}')
else:
    raise FileNotFoundError(
        f'Need {LGBM_OOF_PATH} or {FEATURES_CACHE}+lightgbm to proceed. '
        f'Re-run notebook 12 first.')

print(f'\nLightGBM OOF: {len(lgbm_oof)} rows, {lgbm_oof.shape[1]-1} species cols')


## 3 — Align NN + LightGBM OOFs on common filenames and species

**Important scope difference vs notebook 11.** The LightGBM OOF only covers fold-0 files; the NN OOFs cover all 5 folds. We subset the NN OOFs to fold 0 only so the comparison is apples-to-apples. (The classical model wasn't trained across all folds — that's a 5x cost we deferred.) This means the val set for the stacker shootout is the ~7k fold-0 files, which is enough for stable stacker fits.


In [ ]:
# Subset NN OOFs to fold-0 files only
folds_df = pd.read_csv(FOLDS_CSV)
fold0_files = set(folds_df.loc[folds_df['fold'] == FOLD_FOR_VAL, 'filename'])
loaded_nn_fold0 = {n: d[d['filename'].isin(fold0_files)].copy() for n, d in loaded_nn.items()}
for n, d in loaded_nn_fold0.items():
    print(f'  {n:<14} fold-0 rows: {len(d)}')

# Common filenames across NN + LGBM
all_models = {**loaded_nn_fold0, 'lightgbm': lgbm_oof}
common = set.intersection(*(set(d['filename']) for d in all_models.values()))
print(f'common files across {len(all_models)} models: {len(common)}')

# Canonical species set (majority vote — same logic as notebook 11)
per_model_cols = {n: [c for c in d.columns if c != 'filename'] for n, d in all_models.items()}
set_counts = Counter(frozenset(c) for c in per_model_cols.values())
canonical = set(set_counts.most_common(1)[0][0])
species_cols = sorted(canonical)
print(f'canonical species set: {len(species_cols)} cols (held by {set_counts.most_common(1)[0][1]} models)')

# Align each model to the canonical set. For LGBM, fill missing species with 0.5.
aligned = {}
for n, d in all_models.items():
    d = d[d['filename'].isin(common)].sort_values('filename').reset_index(drop=True)
    cols = [c for c in d.columns if c != 'filename']
    if set(cols) == canonical:
        aligned[n] = d[['filename'] + species_cols]
        continue
    # Missing species → 0.5; extras → drop
    out = pd.DataFrame({'filename': d['filename']})
    for sp in species_cols:
        out[sp] = d[sp].values if sp in d.columns else 0.5
    aligned[n] = out
    missing = sorted(canonical - set(cols))
    extras  = sorted(set(cols) - canonical)
    print(f'  reconciled {n}: filled {len(missing)} missing species with 0.5, '
          f'dropped {len(extras)} extras')

# Build the canonical y_true and per-model prediction tensors
meta = pd.read_csv(META_CSV)
filenames = aligned[next(iter(aligned))]['filename'].values
merged = pd.DataFrame({'filename': filenames}).merge(
    meta[['filename', 'primary_label']], on='filename', how='left')
y_true = np.zeros((len(filenames), len(species_cols)), dtype=np.float32)
for i, sp in enumerate(species_cols):
    y_true[:, i] = (merged['primary_label'] == sp).astype(float)
preds = {n: aligned[n][species_cols].values.astype(np.float32) for n in aligned}

print(f'\nfinal shapes  y_true: {y_true.shape}  preds[*]: {preds[next(iter(preds))].shape}')
print(f'positives per class:  min={y_true.sum(0).min():.0f}  '
      f'median={int(np.median(y_true.sum(0)))}  max={y_true.sum(0).max():.0f}')


## 4 — Stacker harness (lifted from notebook 11)

Three free blends (mean, rank-mean, geomean) score on the full set with no parameters.
Three learned methods (weighted, per-class isotonic, per-class LR) use a 70/30 split — and we repeat across N_SEEDS to get a stable lift estimate.


In [ ]:
def run_stackers(P_stack, y, model_names, seed):
    """
    Returns dict of {method_name: val_macro_auc} for a single seed.
    P_stack: (M, N, C) array of model predictions.
    """
    rng = np.random.default_rng(seed)
    N = y.shape[0]
    perm = rng.permutation(N)
    cut = int(0.7 * N)
    tr_idx, va_idx = perm[:cut], perm[cut:]
    y_tr, y_va = y[tr_idx], y[va_idx]
    P_tr = P_stack[:, tr_idx, :]
    P_va = P_stack[:, va_idx, :]

    out = {}

    # Free blends (evaluated on val portion only, for fair comparison)
    out['mean'], _      = macro_auc_skip_empty(y_va, P_va.mean(0))
    rank_va = np.stack([rank_normalize(P_va[m]) for m in range(P_va.shape[0])])
    out['rank_mean'], _ = macro_auc_skip_empty(y_va, rank_va.mean(0))
    out['geomean'], _   = macro_auc_skip_empty(
        y_va, np.exp(np.log(np.clip(P_va, EPS, 1-EPS)).mean(0)))
    out['best_single']  = max(
        macro_auc_skip_empty(y_va, P_va[m])[0] for m in range(P_va.shape[0]))

    # Optimized weighted (softmax-reparameterized so weights are on the simplex)
    def softmax(x): e = np.exp(x - x.max()); return e / e.sum()
    def neg_auc(theta):
        w = softmax(theta)
        blend = (P_tr * w[:, None, None]).sum(0)
        auc, _ = macro_auc_skip_empty(y_tr, blend)
        return -auc
    res = minimize(neg_auc, np.zeros(P_stack.shape[0]),
                   method='Nelder-Mead', options={'maxiter': 300, 'xatol': 1e-3})
    w_opt = softmax(res.x)
    out['weighted'], _ = macro_auc_skip_empty(y_va, (P_va * w_opt[:, None, None]).sum(0))

    # Per-class isotonic + mean
    P_cal = P_va.copy()
    for m in range(P_tr.shape[0]):
        for c in range(P_tr.shape[2]):
            yt = y_tr[:, c]
            if 2 <= yt.sum() <= len(yt) - 2:
                iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
                iso.fit(P_tr[m, :, c], yt)
                P_cal[m, :, c] = iso.predict(P_va[m, :, c])
    out['isotonic_mean'], _ = macro_auc_skip_empty(y_va, P_cal.mean(0))

    # Per-class logistic regression
    blend_lr = np.zeros_like(P_va[0])
    for c in range(P_tr.shape[2]):
        yt = y_tr[:, c]
        if yt.sum() < 10:
            blend_lr[:, c] = P_va[:, :, c].mean(0)
            continue
        clf = LogisticRegression(max_iter=200, C=1.0)
        clf.fit(P_tr[:, :, c].T, yt)
        blend_lr[:, c] = clf.predict_proba(P_va[:, :, c].T)[:, 1]
    out['logreg_stacker'], _ = macro_auc_skip_empty(y_va, blend_lr)

    return out, w_opt


## 5 — Run stackers WITHOUT and WITH LightGBM, 3 seeds each

In [ ]:
nn_names = list(loaded_nn_fold0.keys())
lgbm_names = nn_names + ['lightgbm']

P_nn   = np.stack([preds[n] for n in nn_names], axis=0)
P_all  = np.stack([preds[n] for n in lgbm_names], axis=0)

print(f'NN-only stack: {P_nn.shape}    NN+LGBM stack: {P_all.shape}\n')

results_nn, results_all = [], []
weights_log = []
for seed in range(N_SEEDS):
    print(f'  seed {seed}...')
    r_nn,  w_nn  = run_stackers(P_nn,  y_true, nn_names,   seed=seed)
    r_all, w_all = run_stackers(P_all, y_true, lgbm_names, seed=seed)
    r_nn['seed']  = seed; r_all['seed'] = seed
    results_nn.append(r_nn); results_all.append(r_all)
    weights_log.append({'seed': seed,
                        **{f'w_nn_{n}': w for n, w in zip(nn_names, w_nn)},
                        **{f'w_all_{n}': w for n, w in zip(lgbm_names, w_all)}})

df_nn  = pd.DataFrame(results_nn).set_index('seed')
df_all = pd.DataFrame(results_all).set_index('seed')

print('\nNN-only macro AUC by seed:')
print(df_nn.round(4).to_string())
print('\nNN+LGBM macro AUC by seed:')
print(df_all.round(4).to_string())


## 6 — Lift table

Mean and std of (`with_lgbm` − `without_lgbm`) per stacker method, across the 3 seeds. The decision criterion from the header: mean lift ≥ 0.001 is a real signal; ≥ 0.003 is worth deploying.


In [ ]:
lift = (df_all - df_nn).rename(columns=lambda c: f'lift_{c}')
summary = pd.DataFrame({
    'auc_nn_mean':   df_nn.mean(),
    'auc_all_mean':  df_all.mean(),
    'lift_mean':     (df_all - df_nn).mean(),
    'lift_std':      (df_all - df_nn).std(),
})
summary['verdict'] = summary['lift_mean'].apply(
    lambda x: 'DEPLOY' if x >= 0.003 else ('REAL SIGNAL' if x >= 0.001 else 'noise'))
summary = summary.sort_values('lift_mean', ascending=False)
print(summary.round({'auc_nn_mean': 4, 'auc_all_mean': 4,
                     'lift_mean': 4, 'lift_std': 4}).to_string())

print(f'\nOptimizer-assigned weight to LightGBM across seeds:')
for row in weights_log:
    print(f'  seed {row["seed"]}:  w_lightgbm = {row["w_all_lightgbm"]:.3f}')


In [ ]:
# Bar plot of lift per method
fig, ax = plt.subplots(figsize=(9, 4.5))
summary_plot = summary.sort_values('lift_mean')
colors = ['#dc2626' if v == 'noise' else '#f59e0b' if v == 'REAL SIGNAL' else '#10b981'
          for v in summary_plot['verdict']]
ax.barh(summary_plot.index, summary_plot['lift_mean'] * 1000,
        xerr=summary_plot['lift_std'] * 1000,
        color=colors, edgecolor='black', linewidth=0.4, capsize=4)
ax.axvline(1.0, linestyle='--', color='#f59e0b', alpha=0.6, label='real-signal threshold (+0.001)')
ax.axvline(3.0, linestyle='--', color='#10b981', alpha=0.6, label='deploy threshold (+0.003)')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('val macro AUC lift from adding LightGBM (×1000, mean ± std over 3 seeds)')
ax.set_title('Stacker lift: with-LGBM minus without-LGBM')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()


## 7 — How to read the result

**If `rank_mean` shows the biggest lift** (the most likely outcome given the calibration mismatch between LGBM's `predict_proba` outputs and the NN softmaxes): ship rank-mean as your blend in the submission notebook. It's literally three lines of code and benefits from LightGBM's diversity without any tuning risk.

**If `logreg_stacker` wins by a clear margin** (≥0.001 over `rank_mean`): ship the per-class LR stacker. It's a 5-coefficient model per species (one weight per base model + intercept), trivial to serialize, and adapts the blend per species so noisy-on-some-species models like LightGBM can be down-weighted exactly where they hurt.

**If `weighted` assigns LightGBM > 0.10**: that's a strong signal the diversity argument is real — the optimizer is voluntarily giving meaningful weight to a 9-AUC-point-weaker model. Conversely, a weight near zero means even the optimizer thinks LGBM doesn't help.

**If all methods come back `noise`**: the 9-point gap was simply too large for the low correlation to overcome at this scale. Don't add LightGBM. Go to notebook 14 and put the GPU time into a new NN backbone instead.

**Caveat on the comparison.** This experiment uses fold-0 NN OOF only, not the full 5-fold concat the original notebook 11 used. Single-fold val AUC is noisier than 5-fold OOF. Any lift you see should ideally be confirmed on the full 5-fold setup before being shipped to LB. The 3-seed averaging here helps, but per-fold variance is a different beast.
